# Step 3: Building the Time-Aware Travel Matrix

In this notebook, we will create a crucial component for our simulation: a pre-computed, time-aware travel matrix. This matrix will provide realistic, data-driven estimates for the distance and duration of any potential trip between two zones in Manhattan, at a specific time.

This approach is vastly superior to naive distance calculations because it is:
1.  **Data-Driven**: Based on thousands of real trips.
2.  **Time-Aware**: Captures the impact of traffic at different days and hours.
3.  **Computationally Efficient**: Allows for fast lookups within our simulation.

In [1]:
import pandas as pd
import numpy as np

### Load the Cleaned Manhattan Dataset

In [2]:
DATA_PATH = 'data/yellow_tripdata_2025-01_manhattan.parquet'
df = pd.read_parquet(DATA_PATH)

# Ensure datetime columns are correctly typed
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

# Create time features
df['day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek
df['hour_of_day'] = df['tpep_pickup_datetime'].dt.hour

df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration,day_of_week,hour_of_day
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,3.00,0.0,1.0,18.00,2.5,0.0,0.0,8.350000,2,0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2.550000,2,0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,2.00,0.0,1.0,12.10,2.5,0.0,0.0,1.950000,2,0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,0.00,0.0,1.0,9.70,0.0,0.0,0.0,5.566667,2,0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,0.00,0.0,1.0,8.30,0.0,0.0,0.0,3.533333,2,0


### Aggregate Data to Create the Matrix

Now, we'll group by origin, destination, and time to calculate the average trip distance and duration for each route segment.

In [3]:
travel_matrix = df.groupby(['PULocationID', 'DOLocationID', 'day_of_week', 'hour_of_day']).agg(
    mean_distance=('trip_distance', 'mean'),
    mean_duration=('trip_duration', 'mean')
).reset_index()

print(f"Created travel matrix with {len(travel_matrix)} entries.")
travel_matrix.head()

Created travel matrix with 303928 entries.


,PULocationID,DOLocationID,day_of_week,hour_of_day,mean_distance,mean_duration
0,4,4,0,0,0.53,3.433333
1,4,4,0,12,0.18,3.150000
2,4,4,1,2,0.21,2.316667
3,4,4,1,13,1.10,12.416667
4,4,4,1,23,0.45,3.516667


### Save the Travel Matrix

Finally, we save this matrix to a Parquet file. This is our final pre-computation step. The environment will load this file to power its reward calculations.

In [4]:
OUTPUT_PATH = 'data/travel_matrix.parquet'
travel_matrix.to_parquet(OUTPUT_PATH)

print(f"Travel matrix saved to {OUTPUT_PATH}")

Travel matrix saved to data/travel_matrix.parquet
